# Iceberg Table — Spark via Apache Polaris Catalog

**Notebook**: `03_load_iceberg_data.ipynb`  
**Purpose**: Connect to the Spark cluster with the Polaris REST catalog, verify
the `lakehouse.events` Iceberg table, and load synthetic rows for testing.

## Pre-requisites
```
pip install pyspark==3.5.* pyiceberg ipykernel
```
Set the following environment variables before starting JupyterLab:
```bash
export POLARIS_URL='http://polaris:8181/api/catalog'
export POLARIS_CREDENTIAL='spark-client:changeme'
export AWS_ACCESS_KEY_ID='<read from OpenBao: secret/platform/s3 -> access_key>'
export AWS_SECRET_ACCESS_KEY='<read from OpenBao: secret/platform/s3 -> secret_key>'
export AWS_REGION='us-east-2'
export S3_BUCKET='xdatatoiceberg1'
export RBAC_TOKEN='<your-rbac-plane-token>'
export RBAC_URL='http://rbac-plane.prod.svc.cluster.local:8080'
export SPARK_SUBMITTER='<your-username>'  # used for RBAC gate
```

## 1. RBAC Permission Check
Verify the calling user has `iceberg_engineer` role before doing any work.

In [ ]:
import os, sys, httpx

RBAC_URL   = os.environ.get('RBAC_URL',   'http://localhost:8080')
RBAC_TOKEN = os.environ['RBAC_TOKEN']
CALLER     = os.environ.get('SPARK_SUBMITTER', os.environ.get('USER', 'unknown'))

def rbac_check(username: str, required_perms: set[str]) -> None:
    resp = httpx.get(
        f'{RBAC_URL}/api/v1/users/{username}/roles',
        headers={'Authorization': f'Bearer {RBAC_TOKEN}'},
        timeout=10,
    )
    resp.raise_for_status()
    data  = resp.json()
    perms = {f"{p['service']}:{p['permission']}" for p in data.get('permissions', [])}
    missing = required_perms - perms
    if missing:
        raise PermissionError(f'[RBAC] DENIED for {username} — missing: {missing}')
    print(f'[RBAC] ✓  {username} — roles: {data["roles"]}')
    print(f'       effective permissions: {len(perms)}')

rbac_check(CALLER, {
    'spark:USE_CATALOG',
    'spark:WRITE_ICEBERG',
    'spark:SUBMIT_JOB',
    'polaris:TABLE_WRITE',
    'polaris:CATALOG_READ',
})

## 2. Build Spark Session (Polaris REST catalog + S3A)

In [ ]:
from pyspark.sql import SparkSession

S3_BUCKET          = os.environ.get('S3_BUCKET',  'xdatatoiceberg1')
S3_REGION          = os.environ.get('AWS_REGION', 'us-east-2')
POLARIS_URL        = os.environ.get('POLARIS_URL', 'http://polaris:8181/api/catalog')
POLARIS_CREDENTIAL = os.environ.get('POLARIS_CREDENTIAL', 'spark-client:changeme')
AWS_KEY            = os.environ['AWS_ACCESS_KEY_ID']
AWS_SECRET         = os.environ['AWS_SECRET_ACCESS_KEY']
WAREHOUSE          = f's3a://{S3_BUCKET}/warehouse'
TARGET_FILE_BYTES  = int(2.56 * 1024 * 1024)   # 2 684 354 bytes ≈ 2.56 MB

spark = (
    SparkSession.builder
    .appName('IcebergDataLoad')
    # Iceberg extensions
    .config('spark.sql.extensions',
            'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions')
    # Polaris REST catalog
    .config('spark.sql.catalog.polaris',           'org.apache.iceberg.spark.SparkCatalog')
    .config('spark.sql.catalog.polaris.type',      'rest')
    .config('spark.sql.catalog.polaris.uri',       POLARIS_URL)
    .config('spark.sql.catalog.polaris.credential', POLARIS_CREDENTIAL)
    .config('spark.sql.catalog.polaris.warehouse', WAREHOUSE)
    .config('spark.sql.catalog.polaris.io-impl',   'org.apache.iceberg.aws.s3.S3FileIO')
    .config('spark.sql.catalog.polaris.s3.region', S3_REGION)
    .config('spark.sql.catalog.polaris.write.target-file-size-bytes', str(TARGET_FILE_BYTES))
    # S3A credentials
    .config('spark.hadoop.fs.s3a.endpoint',         f'https://s3.{S3_REGION}.amazonaws.com')
    .config('spark.hadoop.fs.s3a.access.key',        AWS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key',        AWS_SECRET)
    .config('spark.hadoop.fs.s3a.path.style.access', 'false')
    .config('spark.hadoop.fs.s3a.impl',              'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)

## 3. Verify the Table Exists

In [ ]:
spark.sql('USE CATALOG polaris').show()
spark.sql('SHOW NAMESPACES').show()
spark.sql('SHOW TABLES IN polaris.lakehouse').show()
spark.sql('DESCRIBE EXTENDED polaris.lakehouse.events').show(50, truncate=False)

## 4. Generate Synthetic Data
Creates 200 000 rows spanning 48 hours → 48 × 4 = 192 partition files.
Each Parquet file is capped at ~2.56 MB via `write.target-file-size-bytes`.

In [ ]:
import random
from datetime import datetime, timedelta, timezone
from pyspark.sql import Row
from pyspark.sql.types import (DoubleType, LongType, StringType,
                                StructField, StructType, TimestampType)
import json

EVENT_TYPES = ['click', 'purchase', 'view', 'add_to_cart', 'checkout']
SOURCES     = ['web', 'mobile_ios', 'mobile_android', 'api']

BASE_TS  = datetime(2024, 6, 1, 0, 0, 0, tzinfo=timezone.utc)
NUM_ROWS = 200_000

def make_row(i: int) -> Row:
    hour_offset  = i % 48
    minute_jitter = random.randint(0, 3599)
    ts = BASE_TS + timedelta(hours=hour_offset, seconds=minute_jitter)
    return Row(
        event_id   = i,
        source     = random.choice(SOURCES),
        event_type = random.choice(EVENT_TYPES),
        user_id    = random.randint(1, 50_000),
        amount     = round(random.uniform(0.99, 499.99), 2),
        ts         = ts,
        payload    = json.dumps({'session': f's{random.randint(1,9999)}', 'v': 1}),
    )

schema = StructType([
    StructField('event_id',   LongType(),      False),
    StructField('source',     StringType(),    True),
    StructField('event_type', StringType(),    True),
    StructField('user_id',    LongType(),      True),
    StructField('amount',     DoubleType(),    True),
    StructField('ts',         TimestampType(), False),
    StructField('payload',    StringType(),    True),
])

rows = [make_row(i) for i in range(NUM_ROWS)]
df   = spark.createDataFrame(rows, schema=schema)
print(f'Rows generated: {df.count():,}')
df.show(5)

## 5. Write to Iceberg Table
Spark honours `write.target-file-size-bytes` (2.56 MB) and the
`PARTITIONED BY (hours(ts), bucket(4, event_id))` spec from the DDL.

In [ ]:
(
    df.writeTo('polaris.lakehouse.events')
      .option('write-audit-publish', 'false')
      .append()
)
print('Write complete.')

## 6. Verify Rows and Partitions

In [ ]:
total = spark.sql('SELECT COUNT(*) AS cnt FROM polaris.lakehouse.events').collect()[0]['cnt']
print(f'Total rows in table: {total:,}')

spark.sql("""
    SELECT
        date_trunc('hour', ts) AS hour_bucket,
        event_id % 4           AS hash_bucket,
        COUNT(*)               AS row_count
    FROM polaris.lakehouse.events
    GROUP BY 1, 2
    ORDER BY 1, 2
    LIMIT 20
""").show(truncate=False)

# Show snapshot history
spark.sql('SELECT snapshot_id, committed_at, operation FROM polaris.lakehouse.events.snapshots').show()

## 7. Inspect Parquet File Sizes on S3
Confirm files are approximately 2.56 MB.

In [ ]:
import boto3

s3 = boto3.client(
    's3',
    region_name          = os.environ.get('AWS_REGION', 'us-east-2'),
    aws_access_key_id    = os.environ['AWS_ACCESS_KEY_ID'],
    aws_secret_access_key= os.environ['AWS_SECRET_ACCESS_KEY'],
)

paginator = s3.get_paginator('list_objects_v2')
sizes = []
for page in paginator.paginate(Bucket='xdatatoiceberg1', Prefix='warehouse/lakehouse/events/data/'):
    for obj in page.get('Contents', []):
        if obj['Key'].endswith('.parquet'):
            sizes.append(obj['Size'])

if sizes:
    avg_mb = sum(sizes) / len(sizes) / 1024 / 1024
    print(f'Parquet files found : {len(sizes)}')
    print(f'Average size        : {avg_mb:.2f} MB')
    print(f'Min / Max           : {min(sizes)/1024/1024:.2f} MB / {max(sizes)/1024/1024:.2f} MB')
else:
    print('No parquet files found — check the prefix or wait for the write to flush.')

## 8. Stop Spark

In [ ]:
spark.stop()
print('Spark session stopped.')